In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import json
import numpy as np
from selenium.common.exceptions import StaleElementReferenceException
from selenium.webdriver.chrome.options import Options

In [2]:
## IPV4

driver = webdriver.Chrome()
driver.get('https://www.spamhaus.org/drop/drop_v4.json')
pre_tag = driver.find_element("tag name", "pre")
json_text = pre_tag.text

lines = json_text.strip().split('\n')
data = [json.loads(line) for line in lines]

df = pd.DataFrame(data)

In [3]:
df.head()

,cidr,sblid,rir,type,timestamp,size,records,copyright,terms
0,1.10.16.0/20,SBL256894,apnic,NaN,NaN,NaN,NaN,NaN,NaN
1,1.19.0.0/16,SBL434604,apnic,NaN,NaN,NaN,NaN,NaN,NaN
2,1.32.128.0/18,SBL286275,apnic,NaN,NaN,NaN,NaN,NaN,NaN
3,2.56.192.0/22,SBL459831,ripencc,NaN,NaN,NaN,NaN,NaN,NaN
4,2.57.122.0/24,SBL636050,ripencc,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df = df.drop(columns=['type','timestamp','size','records','copyright','terms'])

In [4]:
df = pd.read_excel('ipv4.xlsx')

In [5]:
sblid1 = np.array(df['sblid'])

In [6]:
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9224")
driver = webdriver.Chrome(options=options)

reason1 = []
for sbl in sblid1:
    box = WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.CSS_SELECTOR,
        "input[class*='rounded-full w-full h-14']")))
    box.send_keys(sbl)

    WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".cursor-pointer"))).click()

    for _ in range(3):
        try:
            text = WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.XPATH,
                '//*[@id="__nuxt"]/main/div[2]/main/div/div/section/div/div[2]/div[2]/div[4]/div/h2[1]'))).text
            reason1.append(text)
            break
        except StaleElementReferenceException:
            continue

    WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.CSS_SELECTOR,
        "input[class*='rounded-full w-full h-14']"))).clear()

In [7]:
df['reason']= reason1

In [8]:
def categorize_reasons_prioritywise(reasons):
    categories = [
        ("Snowshoe Spam", ["snowshoe", "suspected snowshoe"]),
        ("Bulletproof Hosting", ["bulletproof hosting", "cybercrime hub", "dirty hosting", "bad hosting", "escalation"]),
        ("Hijacking", ["hijacked", "ip hijack", "hijack"]),
        ("Brute Force", ["brute-force", "brute force", "password brute forcing"]),
        ("Phishing", ["phishing"]),
        ("Botnet", ["botnet"]),
        ("Cybercrime", ["cybercrime", "crime network"]),
        ("Carding", ["carding"]),
        ("Spam", ["spammer", "spam emitters", "spam", "fake soumu spam", "redspeed"]),
        ("Unallocated IP", ["unallocated ip", "spam from unallocated"]),
    ]

    reason_lower = reasons.lower()
    for category, keywords in categories:
        if any(keyword in reason_lower for keyword in keywords):
            return category
    return "Uncategorized"

df["category"] = df["reason"].apply(categorize_reasons_prioritywise)


In [9]:
rir_to_region = {
    'arin': 'North America',
    'ripencc': 'Europe',
    'apnic': 'Asia-Pacific',
    'lacnic': 'Latin America',
    'afrinic': 'Africa'
}
df['region'] = df['rir'].map(rir_to_region)


In [10]:
import ipaddress

def get_first_ip(ip_range):
    try:
        return str(ipaddress.ip_network(ip_range, strict=False)[0])
    except:
        return None

df['first_ip'] = df['cidr'].apply(get_first_ip)


In [11]:
ip = np.array(df['first_ip'])

In [13]:
import requests, time

orgs1 = [] 
asns1 = []
country1= []
for ip in ip:
    r = requests.get(f"http://ip-api.com/csv/{ip}").text.split(',')
    orgs1.append(r[11] if r[0] == "success" else None)
    asns1.append(r[12] if r[0] == "success" else None)
    country1.append(r[1] if r[0] == "success" else None)
    time.sleep(1)


In [14]:
df['org'] = orgs1
df['asn'] = asns1
df['country'] = country1

In [15]:
df.head()

,cidr,sblid,rir,reason,category,region,first_ip,org,asn,country
0,1.10.16.0/20,SBL256894,apnic,snowshoe range - CHINANET Guangdong province n...,Snowshoe Spam,Asia-Pacific,1.10.16.0,Chinanet GD,,China
1,1.19.0.0/16,SBL434604,apnic,Hijacked IP range - APNIC bogon (unallocated),Hijacking,Asia-Pacific,1.19.0.0,Korea Internet Security Agency,,South Korea
2,1.32.128.0/18,SBL286275,apnic,"Singapore Telecom, ConnectPlus (AS38183)",Uncategorized,Asia-Pacific,1.32.128.0,"""Singapore Telecom","ConnectPlus""",Singapore
3,2.56.192.0/22,SBL459831,ripencc,Suspected Snowshoe Spam IP Range - UAB Ripuva,Snowshoe Spam,Europe,2.56.192.0,DataWeb B.V,,The Netherlands
4,2.57.122.0/24,SBL636050,ripencc,DMZHOSTdotco / PPTECHNOLOGY LIMITED - bulletpr...,Bulletproof Hosting,Europe,2.57.122.0,Techoff SRV Limited,AS47890 UNMANAGED LTD,The Netherlands


In [16]:
df.to_excel('ipv4.xlsx')

IPV6

In [17]:
## IPV6

driver = webdriver.Chrome()
driver.get('https://www.spamhaus.org/drop/drop_v6.json')
pre_tag = driver.find_element("tag name", "pre")
json_text = pre_tag.text

# Split the lines and parse each line
lines = json_text.strip().split('\n')
data = [json.loads(line) for line in lines]

# Convert to DataFrame
df2 = pd.DataFrame(data)

In [18]:
df2=df2.drop(columns=['type','timestamp','size','records','copyright','terms'])

In [19]:
df2.head()

,cidr,sblid,rir
0,2001:678:6c4::/48,SBL626637,ripencc
1,2001:678:724::/48,SBL631364,ripencc
2,2001:678:738::/48,SBL635837,ripencc
3,2001:678:73c::/48,SBL618306,ripencc
4,2001:678:754::/48,SBL430383,ripencc


In [20]:
df2 = df2.drop(index=77)

In [23]:
df2 = pd.read_excel('ipv6.xlsx')

In [24]:
sblid2 = np.array(df2['sblid'])

In [25]:
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9224")
driver = webdriver.Chrome(options=options)

reason2 = []
for sbl in sblid2:
    box = WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.CSS_SELECTOR,
        "input[class*='rounded-full w-full h-14']")))
    box.send_keys(sbl)

    WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".cursor-pointer"))).click()

    for _ in range(3):
        try:
            text = WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.XPATH,
                '//*[@id="__nuxt"]/main/div[2]/main/div/div/section/div/div[2]/div[2]/div[4]/div/h2[1]'))).text
            reason2.append(text)
            break
        except StaleElementReferenceException:
            continue

    WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.CSS_SELECTOR,
        "input[class*='rounded-full w-full h-14']"))).clear()

In [26]:
df2['reason']=reason2

In [27]:
def categorize_reason(reason):
    categories = [
        ("Snowshoe Spam", ["snowshoe", "suspected snowshoe"]),
        ("Bulletproof Hosting", ["bulletproof hosting", "cybercrime hub", "dirty hosting", "bad hosting", "escalation"]),
        ("Hijacking", ["hijacked", "ip hijack", "hijack"]),
        ("Brute Force", ["brute-force", "brute force", "password brute forcing"]),
        ("Phishing", ["phishing"]),
        ("Botnet", ["botnet"]),
        ("Cybercrime", ["cybercrime", "crime network"]),
        ("Carding", ["carding"]),
        ("Spam", ["spammer", "spam emitters", "spam", "fake soumu spam", "redspeed"]),
        ("Unallocated IP", ["unallocated ip", "spam from unallocated"]),
    ]

    reason_lower = reason.lower()
    for category, keywords in categories:
        if any(keyword in reason_lower for keyword in keywords):
            return category
    return "Uncategorized"

df2["category"] = df2["reason"].apply(categorize_reason)


In [28]:
rir_to_region = {
    'arin': 'North America',
    'ripencc': 'Europe',
    'apnic': 'Asia-Pacific',
    'lacnic': 'Latin America',
    'afrinic': 'Africa'
}
df2['region'] = df2['rir'].map(rir_to_region)


In [29]:
import ipaddress

def get_first_ip(ip_range):
    try:
        return str(ipaddress.ip_network(ip_range, strict=False)[0])
    except:
        return None

df2['first_ip'] = df2['cidr'].apply(get_first_ip)


In [30]:
ip2 = np.array(df2['first_ip'])

In [31]:
import requests, time

orgs2 = [] 
asns2 = []
country2= []
for ip in ip2:
    r = requests.get(f"http://ip-api.com/csv/{ip}").text.split(',')
    orgs2.append(r[11] if r[0] == "success" else None)
    asns2.append(r[12] if r[0] == "success" else None)
    country2.append(r[1] if r[0] == "success" else None)
    time.sleep(1)


In [32]:
df2['org']= orgs2
df2['asn'] = asns2
df2['country'] = country2

In [33]:
df2.head()

,cidr,sblid,rir,reason,category,region,first_ip,org,asn,country
0,2001:678:6c4::/48,SBL626637,ripencc,Suspected Snowshoe Spam IP Range - EL.K.STIL LLC,Snowshoe Spam,Europe,2001:678:6c4::,EL.K.STIL LLC,AS202383 EL.K.STIL LLC,Ukraine
1,2001:678:724::/48,SBL631364,ripencc,Suspected Snowshoe Spam IP Range - EL.K.STIL LLC,Snowshoe Spam,Europe,2001:678:724::,"""ALTERNATIVE CREDIT CENTRE """"EXPRESS CREDIT"""" ...",AS202383 EL.K.STIL LLC,Ukraine
2,2001:678:738::/48,SBL635837,ripencc,Suspected Snowshoe Spam IP Range - PP DOBROSVI...,Snowshoe Spam,Europe,2001:678:738::,PP DOBROSVIT-KREDO,AS202267 PP DOBROSVIT-KREDO,Ukraine
3,2001:678:73c::/48,SBL618306,ripencc,Suspected Snowshoe Spam IP Range - GROMADSKA S...,Snowshoe Spam,Europe,2001:678:73c::,Gromadska Spilka Ukrainy LLC,AS202266 GROMADSKA SPILKA UKRAINY LLC,Ukraine
4,2001:678:754::/48,SBL430383,ripencc,"Suspected Snowshoe Spam IP Range - PP ""TORERO""",Snowshoe Spam,Europe,2001:678:754::,"""PP """"TORERO""""""",,France


In [34]:
df2.to_excel('ipv6.xlsx')

In [36]:
df['asn']=df['asn'].replace({np.nan:"Unknown"})

In [37]:
df['org']=df['org'].replace({np.nan:"Unknown"})

In [44]:
df['asn'] = df['asn'].str.split().str[0]

In [45]:
df['asn'] = df['asn'].str.strip('"')

In [38]:
df2['org']=df2['org'].str.strip('"')

In [40]:
df2['asn'] = df2['asn'].str.split().str[0]

In [39]:
df2['org'] = df2['org'].replace({np.nan:'Unknown'})
df2['asn'] = df2['asn'].replace({np.nan:'Unknown'})

In [47]:
df.to_excel('ipv4.xlsx',index=0)
df2.to_excel('ipv6.xlsx',index=0)